In [1]:
import os
import sys
import traceback
import warnings
warnings.filterwarnings('ignore', 'GeoSeries.notna', UserWarning)
sys.path.append("../src")

from connection_util import DatabaseConnection
from validation_util import Validation
from savepoints_util import SavePoints
from etl_pipeline_chameleon import run_chameleon_etl_pipeline
from etl_pipeline_project_level import run_project_level_etl_pipeline
from version import create_version
from cache_util import clear_cache

In [2]:
ETL_ROLE = 'gis_update' 

target_username = os.getenv(f'{ETL_ROLE.upper()}_USERNAME')
target_password = os.getenv(f'{ETL_ROLE.upper()}_PASSWORD')


In [3]:
# val_host = os.getenv("VAL_HOST")
# val_database_name = os.getenv("VAL_DATABASE_NAME")
# val_port = os.getenv("VAL_PORT")
# val_username = os.getenv("VAL_USERNAME")
# val_password = os.getenv("VAL_PASSWORD")


# col_host = os.getenv("COL_HOST")
# col_database_name = os.getenv("COL_DATABASE_NAME")
# col_port = os.getenv("COL_PORT")
# col_username = os.getenv("COL_USERNAME")
# col_password = os.getenv("COL_PASSWORD")


# buf_host = os.getenv("BUF_HOST")
# buf_database_name = os.getenv("BUF_DATABASE_NAME")
# buf_port = os.getenv("BUF_PORT")
# buf_username = os.getenv("BUF_USERNAME")
# buf_password = os.getenv("BUF_PASSWORD")

# akw_host = os.getenv("AKW_HOST")
# akw_database_name = os.getenv("AKW_DATABASE_NAME")
# akw_port = os.getenv("AKW_PORT")
# akw_username = os.getenv("AKW_USERNAME")
# akw_password = os.getenv("AKW_PASSWORD")


source_host = os.getenv("SOURCE_HOST")
source_database_name = os.getenv("SOURCE_DATABASE_NAME")
source_port = os.getenv("SOURCE_PORT")
source_username = os.getenv("SOURCE_USERNAME")
source_password = os.getenv("SOURCE_PASSWORD")

chameleon_target_host = os.getenv("CHAMELEON_TARGET_HOST")
chameleon_target_database_name = os.getenv("CHAMELEON_TARGET_DATABASE_NAME")
chameleon_target_port = os.getenv("CHAMELEON_TARGET_PORT")
chameleon_target_username = target_username
chameleon_target_password = target_password

project_level_target_host = os.getenv("PROJECT_LEVEL_TARGET_HOST")
project_level_target_database_name = os.getenv("PROJECT_LEVEL_TARGET_DATABASE_NAME")
project_level_target_port = os.getenv("PROJECT_LEVEL_TARGET_PORT")
project_level_target_username = target_username
project_level_target_password = target_password

In [4]:
# val_conn = DatabaseConnection.connect_to_validation_postgis_database(val_host,val_database_name,val_port,val_username,val_password)
source_conn = DatabaseConnection.connect_to_source_postgis_database(source_host,source_database_name,source_port,source_username,source_password)
chameleon_target_conn= DatabaseConnection.connect_to_target_postgis_database(chameleon_target_host,chameleon_target_database_name,chameleon_target_port,chameleon_target_username,chameleon_target_password)
project_level_target_conn= DatabaseConnection.connect_to_target_postgis_database(project_level_target_host,project_level_target_database_name,project_level_target_port,project_level_target_username,project_level_target_password)



Source database connection successful!
Target database connection successful!
Target database connection successful!


In [5]:

# SCHEMA_NAMES = {'akwaaba':'akwaaba_schema','buffalo':'buffalo_schema','colobus':'colobus_schema'}
SOURCE_SCHEMA_NAMES = {
    'akwaaba':['akwaaba_schema'],
    'buffalo':['buffalo_schema_baseline','buffalo_schema_operational'],
    'colobus':['colobus_schema']
}

TARGET_SCHEMA_NAMES = {
    'akwaaba':['akwaaba_schema'],
    'buffalo':['buffalo_schema', 'buffalo_schema_baseline','buffalo_schema_operational'],
    'colobus':['colobus_schema']
    }
CLEAR_CACHE = True


In [6]:

def main_chameleon_etl():
    try:

        # # Extract, transform and load  into Chameleon database
       
       #Akwaaba    
        run_chameleon_etl_pipeline(SOURCE_SCHEMA_NAMES['akwaaba'][0],TARGET_SCHEMA_NAMES['akwaaba'][0], source_conn, chameleon_target_conn)
  
        
        # # #Buffalo 
        run_chameleon_etl_pipeline(SOURCE_SCHEMA_NAMES['buffalo'][0],TARGET_SCHEMA_NAMES['buffalo'][1], source_conn, chameleon_target_conn)
        run_chameleon_etl_pipeline(SOURCE_SCHEMA_NAMES['buffalo'][1],TARGET_SCHEMA_NAMES['buffalo'][2], source_conn, chameleon_target_conn)
        run_chameleon_etl_pipeline(SOURCE_SCHEMA_NAMES['buffalo'][1],TARGET_SCHEMA_NAMES['buffalo'][0], source_conn, chameleon_target_conn)
        
        # # Colobus
        run_chameleon_etl_pipeline(SOURCE_SCHEMA_NAMES['colobus'][0],TARGET_SCHEMA_NAMES['colobus'][0], source_conn, chameleon_target_conn)
       


        # RUn Backup
        create_version('Added year requested and agreement type to lease layer')

        # Clear cache — use one of:
        # clear_cache()                                          # all projects
        # clear_cache('buffalo')                                 # one project
        # clear_cache(['akwaaba', 'buffalo', 'colobus'])         # multiple projects
        

        print('All Layers are Loaded')

    except Exception as e:
        print("ETL Failed with error message", e)
        traceback.print_exc()


main_chameleon_etl()


All geometries valid: True (0 remaining invalid)
All geometries valid: True (0 remaining invalid)
All geometries valid: True (0 remaining invalid)
Successfully saved 84 records to akwaaba_schema.compartments
akwaaba_schema compartments into akwaaba_schema Loaded Successfully
Successfully saved 14 records to akwaaba_schema.leased_areas
akwaaba_schema leased Areas into akwaaba_schema Loaded Successfully
All geometries valid: True (0 remaining invalid)
akwaaba_schema loading unplantable_areas...
Successfully saved 673 records to akwaaba_schema.subcompartments
akwaaba_schema plantable_Areas into akwaaba_schema Loaded Successfully
akwaaba_schema unplantable_areas: 185 records
akwaaba_schema loading protected_areas...
akwaaba_schema protected_areas: 261 records
akwaaba_schema merging layers: subcompartments(671), unplantable_areas(185), protected_areas(261)
All geometries valid: True (0 remaining invalid)


c:\Users\LENOVO\Desktop\etl_pipeline\notebook\../src\etl_util_chameleon.py:615: UserWarning: Geometry is in a geographic CRS. Results from 'area' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  overlay_result['_area'] = overlay_result.geometry.area


Successfully saved 154 records to akwaaba_schema.rb_roads
akwaaba_schema Rb Roads Loaded into akwaaba_schema Loaded Successfully
Fixing 53 invalid geometries...
All geometries valid: True (0 remaining invalid)
Successfully saved 1117 records to akwaaba_schema.lease_areas_stratum
akwaaba_schema Lease Layer Stratum Snapped Loaded Successfully
akwaaba_schema Lease Layer Stratum Loaded Successfully
All geometries valid: True (0 remaining invalid)
All geometries valid: True (0 remaining invalid)
All geometries valid: True (0 remaining invalid)
Successfully saved 107 records to buffalo_schema_baseline.leased_areas
buffalo_schema_baseline leased Areas into buffalo_schema_baseline Loaded Successfully
All geometries valid: True (0 remaining invalid)
Successfully saved 219 records to buffalo_schema_baseline.compartments
buffalo_schema_baseline compartments into buffalo_schema_baseline Loaded Successfully
Fixing 1 invalid geometries...
All geometries valid: True (0 remaining invalid)
Successfully

In [7]:

def main_project_level_etl():
    try:

      

        # # Extract, transform and load  into Project Level database
    #    #Akwaaba    
        run_project_level_etl_pipeline(SOURCE_SCHEMA_NAMES['akwaaba'][0],TARGET_SCHEMA_NAMES['akwaaba'][0], source_conn, project_level_target_conn)
        
    #     #Buffalo 
        run_project_level_etl_pipeline(SOURCE_SCHEMA_NAMES['buffalo'][0],TARGET_SCHEMA_NAMES['buffalo'][1], source_conn, project_level_target_conn)
        run_project_level_etl_pipeline(SOURCE_SCHEMA_NAMES['buffalo'][1],TARGET_SCHEMA_NAMES['buffalo'][2], source_conn, project_level_target_conn)
        run_project_level_etl_pipeline(SOURCE_SCHEMA_NAMES['buffalo'][1],TARGET_SCHEMA_NAMES['buffalo'][0], source_conn, project_level_target_conn)
        
        # # Colobus
        run_project_level_etl_pipeline(SOURCE_SCHEMA_NAMES['colobus'][0],TARGET_SCHEMA_NAMES['colobus'][0], source_conn, project_level_target_conn)


        # RUn Backup
        # create_version('fixed self intersection in subcompartment')


        # Clear cache — use one of:
        # clear_cache()                                          # all projects
        # clear_cache('buffalo')                                 # one project
        # clear_cache(['akwaaba', 'buffalo', 'colobus'])         # multiple projects
        

        print('All Layers are Loaded')

    except Exception as e:
        print("ETL Failed with error message", e)
        traceback.print_exc()


main_project_level_etl()


All geometries valid: True (0 remaining invalid)
All geometries valid: True (0 remaining invalid)
All geometries valid: True (0 remaining invalid)
Successfully saved 154 records to akwaaba_schema.rb_roads
akwaaba_schema Rb Roads Loaded into akwaaba_schema Loaded Successfully
Successfully saved 14 records to akwaaba_schema.leased_areas
akwaaba_schema leased Areas into akwaaba_schema Loaded Successfully
Successfully saved 84 records to akwaaba_schema.compartments
akwaaba_schema compartments into akwaaba_schema Loaded Successfully
akwaaba_schema loading unplantable_areas...
akwaaba_schema unplantable_areas: 185 records
akwaaba_schema loading protected_areas...
akwaaba_schema protected_areas: 261 records
akwaaba_schema merging layers: subcompartments(671), unplantable_areas(185), protected_areas(261)
All geometries valid: True (0 remaining invalid)
Fixing 53 invalid geometries...
All geometries valid: True (0 remaining invalid)
Successfully saved 1117 records to akwaaba_schema.lease_areas_